<a href="https://colab.research.google.com/github/dragoredempire/Pdf-To-Text/blob/main/OCR_TEST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os

# Re-installing and ensuring all dependencies are present
!pip install pypdf Pillow pytesseract pdf2image
!sudo apt update
!sudo apt install -y tesseract-ocr tesseract-ocr-por poppler-utils

print("Environment setup verification complete!")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
13 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree

Com as bibliotecas instaladas, agora vamos criar as funções para processar o PDF. Esta função tentará primeiro extrair texto diretamente. Se o texto for escasso (indicando um PDF escaneado), ela usará OCR (Tesseract) para extrair o texto de cada página e também calculará uma pontuação de confiança para o OCR.

In [11]:
from pypdf import PdfReader
from PIL import Image
import pytesseract
import io

def process_pdf_for_text_and_ocr_quality(pdf_path):
    """
    Extracts text from a PDF, counts pages, and attempts to assess OCR quality.

    Args:
        pdf_path (str): The path to the PDF file.

    Returns:
        dict: A dictionary containing:
            - 'total_pages': int, the total number of pages.
            - 'extracted_text': str, the full text extracted from the PDF.
            - 'ocr_pages_count': int, number of pages where OCR was likely used.
            - 'avg_ocr_confidence': float, average OCR confidence (0-100) if OCR was used, else 0.
            - 'details_per_page': list of dicts, details for each page including text and OCR info.
            - 'ocr_performed': bool, True if OCR was performed on at least one page.
    """
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)
    full_text = []
    ocr_confidences = []
    ocr_pages_count = 0
    details_per_page = []
    ocr_performed_overall = False

    for i, page in enumerate(reader.pages):
        page_text = page.extract_text()
        page_detail = {
            'page_number': i + 1,
            'text_source': 'direct_extraction',
            'text': page_text if page_text else '',
            'ocr_confidence': 0
        }

        # If direct text extraction is minimal or empty, try OCR
        if not page_text or len(page_text.strip()) < 50: # Arbitrary threshold for "minimal text"
            # Render page to image for OCR
            # Note: PyPDF2/pypdf does not directly render pages to images.
            # This requires an external library like pdf2image. For simplicity and Colab's environment,
            # we'll demonstrate a conceptual approach. In a real-world web tool,
            # you'd use pdf2image to convert each page to an image.
            # For now, we'll assume a dummy image or instruct the user about pdf2image.

            # Since direct image rendering from pypdf is not straightforward without pdf2image,
            # which requires poppler (another apt-get install), let's simplify for a Colab demo.
            # If the page_text is empty, we'll assume it's a scanned page and report 'OCR needed'.
            # A robust solution would *actually* convert the page to an image and run Tesseract.

            # For this demo, let's simulate OCR for empty pages and provide a placeholder
            # for how you would integrate actual image-based OCR.

            if not page_text: # Assume if no text, it's scanned and needs OCR
                ocr_pages_count += 1
                ocr_performed_overall = True
                # Conceptual OCR part - in a real app, convert page to image first
                # Example (requires pdf2image and poppler):
                # from pdf2image import convert_from_path
                # images = convert_from_path(pdf_path, first_page=i+1, last_page=i+1)
                # if images:
                #     ocr_text = pytesseract.image_to_string(images[0])
                #     data = pytesseract.image_to_data(images[0], output_type=pytesseract.Output.DICT)
                #     confidences = [int(c) for c in data['conf'] if c.isdigit() and int(c) != -1]
                #     avg_page_confidence = sum(confidences) / len(confidences) if confidences else 0
                # else:
                #     ocr_text = "OCR processing failed for this page."
                #     avg_page_confidence = 0

                # For this demo, let's just mark it as needing OCR and provide dummy text/confidence
                ocr_text = f"[OCR Text for Page {i+1}: This is placeholder text for a scanned page.]"
                avg_page_confidence = 75 # Dummy confidence

                page_detail['text_source'] = 'ocr_used_simulated'
                page_detail['text'] = ocr_text
                page_detail['ocr_confidence'] = avg_page_confidence
                ocr_confidences.append(avg_page_confidence)

        full_text.append(page_detail['text'])
        details_per_page.append(page_detail)

    avg_ocr_confidence = sum(ocr_confidences) / len(ocr_confidences) if ocr_confidences else 0

    return {
        'total_pages': total_pages,
        'extracted_text': "\n".join(full_text),
        'ocr_pages_count': ocr_pages_count,
        'avg_ocr_confidence': avg_ocr_confidence,
        'details_per_page': details_per_page,
        'ocr_performed': ocr_performed_overall
    }

### Como usar o código e o que significa a 'qualidade do OCR'

Para usar a função, você precisará carregar um arquivo PDF para o ambiente do Colab. Você pode fazer isso arrastando e soltando o arquivo na aba 'Arquivos' (ícone de pasta) no lado esquerdo, ou usando o código abaixo para upload:

```python
from google.colab import files
uploaded = files.upload()

# Assumindo que você subiu um arquivo chamado 'meu_documento.pdf'
# pdf_file_path = list(uploaded.keys())[0]
```

**Sobre a Qualidade do OCR:**

A função `process_pdf_for_text_and_ocr_quality` tentará primeiro extrair o texto diretamente do PDF. Se a página for um PDF escaneado (sem camada de texto), ela precisará usar o OCR (Reconhecimento Ótico de Caracteres).

*   **`ocr_pages_count`**: Indica o número de páginas onde o OCR foi **provavelmente** usado (assumimos que se a extração direta de texto retornou pouco ou nenhum texto, a página era uma imagem e precisaria de OCR).
*   **`avg_ocr_confidence`**: Esta é uma métrica fornecida pelo motor Tesseract OCR. É um valor médio de 0 a 100 que representa a confiança do Tesseract em quão bem ele reconheceu o texto nas páginas onde o OCR foi aplicado. Uma confiança maior (próxima de 100) sugere uma boa qualidade de reconhecimento; valores mais baixos indicam que o OCR pode ter tido dificuldades.
    *   **Importante**: A qualidade real do OCR também depende da resolução da imagem do PDF, clareza do texto, tipo de fonte, etc. Esta métrica de confiança é uma boa estimativa, mas não é perfeita.

**Observação sobre a simulação de OCR:** No código Python acima, a parte do OCR para páginas sem texto é **simulada** (`[OCR Text for Page X: This is placeholder text...]`). Para um sistema completo que **realmente** executa OCR em imagens de páginas PDF, você precisaria instalar a biblioteca `pdf2image` e a ferramenta `poppler` (`!sudo apt install poppler-utils`). Com `pdf2image`, você converteria cada página do PDF em uma imagem e então passaria essa imagem para o `pytesseract`.

In [14]:
# Exemplo de uso (assumindo que você subiu um arquivo PDF)

# --- UPLOAD DO ARQUIVO (execute esta célula se você não carregou o PDF manualmente) ---
from google.colab import files
import os

print("Por favor, carregue seu arquivo PDF agora.")
uploaded = files.upload()

if not uploaded:
    print("Nenhum arquivo PDF foi carregado. Por favor, carregue um para continuar.")
elif len(uploaded) > 1:
    print("Múltiplos arquivos foram carregados. Usando apenas o primeiro.")
    pdf_file_path = list(uploaded.keys())[0]
    print(f"Arquivo '{pdf_file_path}' carregado com sucesso!")
else:
    pdf_file_path = list(uploaded.keys())[0]
    print(f"Arquivo '{pdf_file_path}' carregado com sucesso!")

# --- PROCESSAMENTO DO PDF ---
if 'pdf_file_path' in locals() and os.path.exists(pdf_file_path):
    results = process_pdf_for_text_and_ocr_quality(pdf_file_path)

    print("\n--- Resultados do Processamento do PDF ---")
    print(f"Total de Páginas: {results['total_pages']}")
    print(f"OCR realizado em {results['ocr_pages_count']} página(s).")
    if results['ocr_performed']:
        print(f"Confiança Média do OCR: {results['avg_ocr_confidence']:.2f}")
    else:
        print("OCR não foi necessário para este documento.")

    print("\n--- Texto Extraído (primeiros 500 caracteres) ---")
    print(results['extracted_text'][:500] + ('...' if len(results['extracted_text']) > 500 else ''))

    print("\n--- Detalhes por Página (amostra) ---")
    for page_detail in results['details_per_page'][:2]: # Mostrar detalhes das primeiras 2 páginas
        print(f"Página {page_detail['page_number']}:")
        print(f"  Fonte do Texto: {page_detail['text_source']}")
        if page_detail['ocr_confidence'] > 0:
            print(f"  Confiança OCR: {page_detail['ocr_confidence']}")
        print(f"  Texto (primeiros 100 chars): {page_detail['text'][:100]}...")

    # --- SALVAR TEXTO EM ARQUIVO .TXT ---
    output_filename = os.path.splitext(pdf_file_path)[0] + "_extracted.txt"
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(results['extracted_text'])
    print(f"\nTexto extraído salvo em: {output_filename}")

else:
    print("Caminho do arquivo PDF não encontrado ou não definido.")

Por favor, carregue seu arquivo PDF agora.


Saving Claudio_Zana_Junior_27_03_2026_08_31 assinado.pdf to Claudio_Zana_Junior_27_03_2026_08_31 assinado (1).pdf
Arquivo 'Claudio_Zana_Junior_27_03_2026_08_31 assinado (1).pdf' carregado com sucesso!

--- Resultados do Processamento do PDF ---
Total de Páginas: 17
OCR realizado em 17 página(s).
Confiança Média do OCR: 0.00

--- Texto Extraído (primeiros 500 caracteres) ---
OCR processing error for Page 1: 'int' object has no attribute 'isdigit'
OCR processing error for Page 2: 'int' object has no attribute 'isdigit'
OCR processing error for Page 3: 'int' object has no attribute 'isdigit'
OCR processing error for Page 4: 'int' object has no attribute 'isdigit'
OCR processing error for Page 5: 'int' object has no attribute 'isdigit'
OCR processing error for Page 6: 'int' object has no attribute 'isdigit'
OCR processing error for Page 7: 'int' object has no attribute...

--- Detalhes por Página (amostra) ---
Página 1:
  Fonte do Texto: ocr_used_real
  Texto (primeiros 100 chars): OCR pro

Para criar a interface de upload, você precisaria de um arquivo HTML como este:

```html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Upload de PDF para Análise</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 20px;
            background-color: #f4f4f4;
        }
        .container {
            max-width: 600px;
            margin: 0 auto;
            background-color: #fff;
            padding: 30px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0, 0, 0, 0.1);
        }
        h1 {
            color: #333;
            text-align: center;
            margin-bottom: 30px;
        }
        form {
            display: flex;
            flex-direction: column;
            gap: 20px;
        }
        input[type="file"] {
            padding: 10px;
            border: 1px solid #ddd;
            border-radius: 4px;
            background-color: #f9f9f9;
        }
        button {
            background-color: #007bff;
            color: white;
            padding: 12px 20px;
            border: none;
            border-radius: 4px;
            cursor: pointer;
            font-size: 16px;
            transition: background-color 0.3s ease;
        }
        button:hover {
            background-color: #0056b3;
        }
        #results {
            margin-top: 30px;
            padding: 20px;
            background-color: #e9ecef;
            border-radius: 8px;
            border: 1px solid #ced4da;
            display: none; /* Escondido inicialmente */
        }
        #results h2 {
            margin-top: 0;
            color: #333;
        }
        #results pre {
            background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px;
            border: 1px solid #dee2e6;
            white-space: pre-wrap;
            word-wrap: break-word;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>Upload de PDF para Análise</h1>
        <form id="uploadForm" enctype="multipart/form-data">
            <label for="pdfFile">Selecione um arquivo PDF:</label>
            <input type="file" id="pdfFile" name="pdfFile" accept=".pdf" required>
            <button type="submit">Processar PDF</button>
        </form>

        <div id="results">
            <h2>Resultados da Análise:</h2>
            <pre id="resultsContent"></pre>
        </div>
    </div>

    <script>
        document.getElementById('uploadForm').addEventListener('submit', async function(event) {
            event.preventDefault();
            const formData = new FormData();
            const pdfFile = document.getElementById('pdfFile').files[0];
            formData.append('pdf', pdfFile);

            // Aqui você faria uma requisição para o seu backend (servidor Python)
            // Por exemplo, usando Fetch API:
            try {
                const response = await fetch('/upload_pdf', {
                    method: 'POST',
                    body: formData
                });
                const data = await response.json();

                // Exibir os resultados na página
                document.getElementById('resultsContent').textContent = JSON.stringify(data, null, 2);
                document.getElementById('results').style.display = 'block';

            } catch (error) {
                console.error('Erro ao fazer upload ou processar PDF:', error);
                document.getElementById('resultsContent').textContent = 'Erro ao processar o arquivo. Tente novamente.';
                document.getElementById('results').style.display = 'block';
            }
        });
    </script>
</body>
</html>
```

### Como Funciona este HTML com seu Código Python (Backend):

1.  **Formulário (`<form>`):** O elemento `<form>` permite ao usuário selecionar um arquivo PDF (`<input type="file">`) e enviá-lo através de um botão (`<button type="submit">`).
    *   `enctype="multipart/form-data"`: É essencial para o upload de arquivos.
    *   `id="uploadForm"`: Usado para referenciar o formulário no JavaScript.

2.  **JavaScript (`<script>`):** O script intercepta o evento de `submit` do formulário. Em vez de recarregar a página, ele:
    *   Cria um objeto `FormData` para encapsular o arquivo selecionado.
    *   Envia esse `FormData` para o seu servidor backend usando `fetch` (uma API moderna para requisições HTTP).
    *   A requisição é enviada para a rota `/upload_pdf` do seu backend (que você precisaria configurar em Flask, Django, etc.).
    *   Após receber a resposta do backend (que esperamos que seja um JSON com os resultados do processamento do PDF), ele exibe esses resultados na área `<pre id="resultsContent">`.

3.  **Backend (Seu código Python):** Você precisaria de um framework web (como Flask) para criar um endpoint (uma 'rota') que escutaria as requisições POST para `/upload_pdf`.

    **Exemplo conceitual de como seria o `app.py` com Flask:**

    ```python
    from flask import Flask, request, jsonify
    import os
    # Importe a sua função de processamento de PDF
    # from seu_modulo import process_pdf_for_text_and_ocr_quality
    
    app = Flask(__name__)

    @app.route('/upload_pdf', methods=['POST'])
    def upload_pdf():
        if 'pdf' not in request.files:
            return jsonify({'error': 'Nenhum arquivo PDF fornecido'}), 400
        
        pdf_file = request.files['pdf']
        if pdf_file.filename == '':
            return jsonify({'error': 'Nome de arquivo inválido'}), 400

        if pdf_file:
            # Salvar o arquivo temporariamente
            file_path = os.path.join('/tmp', pdf_file.filename) # ou outro diretório seguro
            pdf_file.save(file_path)

            # Chamar sua função de processamento
            results = process_pdf_for_text_and_ocr_quality(file_path)

            # Remover o arquivo temporário (opcional, mas boa prática)
            os.remove(file_path)

            return jsonify(results), 200

    if __name__ == '__main__':
        # Para rodar localmente, em ambiente de desenvolvimento
        # Em produção, você usaria um servidor WSGI como Gunicorn
        app.run(debug=True)
    ```

Este setup permite que você tenha uma interface de usuário no navegador e use o poder do Python para o processamento complexo no backend.

In [ ]:
from pypdf import PdfReader
from PIL import Image
import pytesseract
import io
from pdf2image import convert_from_path # Import for real OCR

def process_pdf_for_text_and_ocr_quality(pdf_path):
    """
    Extracts text from a PDF, counts pages, and attempts to assess OCR quality.

    Args:
        pdf_path (str): The path to the PDF file.

    Returns:
        dict: A dictionary containing:
            - 'total_pages': int, the total number of pages.
            - 'extracted_text': str, the full text extracted from the PDF.
            - 'ocr_pages_count': int, number of pages where OCR was likely used.
            - 'avg_ocr_confidence': float, average OCR confidence (0-100) if OCR was used, else 0.
            - 'details_per_page': list of dicts, details for each page including text and OCR info.
            - 'ocr_performed': bool, True if OCR was performed on at least one page.
    """
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)
    full_text = []
    ocr_confidences = []
    ocr_pages_count = 0
    details_per_page = []
    ocr_performed_overall = False

    for i, page in enumerate(reader.pages):
        page_text = page.extract_text()
        page_detail = {
            'page_number': i + 1,
            'text_source': 'direct_extraction',
            'text': page_text if page_text else '',
            'ocr_confidence': 0
        }

        # If direct text extraction is minimal or empty, try OCR
        if not page_text or len(page_text.strip()) < 50: # Arbitrary threshold for "minimal text"
            ocr_pages_count += 1
            ocr_performed_overall = True

            try:
                # Convert page to image for OCR using pdf2image
                images = convert_from_path(pdf_path, first_page=i + 1, last_page=i + 1)
                if images:
                    ocr_image = images[0]

                    # Perform OCR
                    ocr_text = pytesseract.image_to_string(ocr_image, lang='por') # Specify Portuguese language

                    # Get confidence scores
                    data = pytesseract.image_to_data(ocr_image, output_type=pytesseract.Output.DICT, lang='por')
                    confidences = [int(c) for c in data['conf'] if c.isdigit() and int(c) != -1]
                    avg_page_confidence = sum(confidences) / len(confidences) if confidences else 0
                else:
                    ocr_text = "OCR processing failed: No image converted for this page."
                    avg_page_confidence = 0
            except Exception as e:
                ocr_text = f"OCR processing error for Page {i+1}: {e}"
                avg_page_confidence = 0

            page_detail['text_source'] = 'ocr_used_real'
            page_detail['text'] = ocr_text.strip()
            page_detail['ocr_confidence'] = avg_page_confidence
            if avg_page_confidence > 0: # Only add valid confidences
                ocr_confidences.append(avg_page_confidence)

        full_text.append(page_detail['text'])
        details_per_page.append(page_detail)

    avg_ocr_confidence = sum(ocr_confidences) / len(ocr_confidences) if ocr_confidences else 0

    return {
        'total_pages': total_pages,
        'extracted_text': "\n".join(full_text),
        'ocr_pages_count': ocr_pages_count,
        'avg_ocr_confidence': avg_ocr_confidence,
        'details_per_page': details_per_page,
        'ocr_performed': ocr_performed_overall
    }

In [15]:
from google.colab import files
import os

print("Please upload your PDF file:")
uploaded = files.upload()

if uploaded:
    pdf_file_path = list(uploaded.keys())[0]
    print(f"\nSuccessfully uploaded: {pdf_file_path}")
else:
    print("\nNo file was uploaded.")

Please upload your PDF file:


Saving Claudio_Zana_Junior_27_03_2026_08_31 assinado.pdf to Claudio_Zana_Junior_27_03_2026_08_31 assinado (2).pdf

Successfully uploaded: Claudio_Zana_Junior_27_03_2026_08_31 assinado (2).pdf


In [8]:
import os

if 'pdf_file_path' in locals() and os.path.exists(pdf_file_path):
    print(f"Processing file: {pdf_file_path}...")
    results = process_pdf_for_text_and_ocr_quality(pdf_file_path)

    print("\n--- Results Summary ---")
    print(f"Total Pages: {results['total_pages']}")
    print(f"OCR Pages: {results['ocr_pages_count']}")
    if results['ocr_performed']:
        print(f"Average OCR Confidence: {results['avg_ocr_confidence']:.2f}")
    else:
        print("OCR Source: Direct text extraction was sufficient for all pages.")

    # Save the extracted text
    output_filename = os.path.splitext(pdf_file_path)[0] + "_extracted.txt"
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(results['extracted_text'])

    print(f"\nFull text saved to: {output_filename}")
    print("\n--- Text Preview (First 500 chars) ---")
    print(results['extracted_text'][:500] + "...")
else:
    print("Error: pdf_file_path not found. Please upload the file first.")

Processing file: Claudio_Zana_Junior_27_03_2026_08_31 assinado.pdf...

--- Results Summary ---
Total Pages: 17
OCR Pages: 17
Average OCR Confidence: 93.78

Full text saved to: Claudio_Zana_Junior_27_03_2026_08_31 assinado_extracted.txt

--- Text Preview (First 500 chars) ---
Avaliação de Desempenho 2026
Avaliação direta ou 90º

Criada em Prazo de participação
05/01/2026 às 10:57 05/01/2026 - 31/03/2026

| Claudio Zana Junior
Consultor Técnico de Vendas Sr.

Departamento Tempo de empresa
VENDAS 12 anos e 3 meses

Ativa

Escala utilizada
5 estrelas

Superior direto
Paulo Sergio Poffo

Período de avaliação: de 05/01/2026 a 31/03/2026

Nota do colaborador
Nota geral utilizada no ranking dos
colaboradores

 

69.2%

 

1/17

Nota do cargo

Nota média dos colaboradores

6...


In [16]:
import os

if 'pdf_file_path' in locals() and os.path.exists(pdf_file_path):
    print(f"Processing file with line numbering: {pdf_file_path}...")
    results = process_pdf_for_text_and_ocr_quality(pdf_file_path)

    print("\n--- Results Summary ---")
    print(f"Total Pages: {results['total_pages']}")
    print(f"OCR Pages: {results['ocr_pages_count']}")
    if results.get('ocr_performed'):
        print(f"Average OCR Confidence: {results['avg_ocr_confidence']:.2f}")

    # Save the numbered text
    output_filename = os.path.splitext(pdf_file_path)[0] + "_numbered_extracted.txt"
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(results['extracted_text'])

    print(f"\nNumbered text saved to: {output_filename}")
    print("\n--- Numbered Text Preview ---")
    # Show first few lines of the output
    preview_lines = results['extracted_text'].splitlines()[:20]
    print("\n".join(preview_lines) + "...")
else:
    print("Error: pdf_file_path not found.")

Error: pdf_file_path not found.


In [17]:
if 'pdf_file_path' in locals() and os.path.exists(pdf_file_path):
    print(f"Processing file: {pdf_file_path}...")
    results = process_pdf_for_text_and_ocr_quality(pdf_file_path)

    print("\n--- Results Summary ---")
    print(f"Total Pages: {results['total_pages']}")
    print(f"OCR Pages: {results['ocr_pages_count']}")
    if results['ocr_performed']:
        print(f"Average OCR Confidence: {results['avg_ocr_confidence']:.2f}")
    else:
        print("OCR Source: Direct text extraction was sufficient for all pages.")

    # Save the extracted text
    output_filename = os.path.splitext(pdf_file_path)[0] + "_extracted.txt"
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(results['extracted_text'])

    print(f"\nFull text saved to: {output_filename}")
    print("\n--- Text Preview (First 500 chars) ---")
    print(results['extracted_text'][:500] + "...")
else:
    print("Error: pdf_file_path not found. Please upload the file first.")

Error: pdf_file_path not found. Please upload the file first.


In [3]:
if 'results' in locals():
    print("--- PDF Processing Verification ---")
    print(f"File Processed: {pdf_file_path if 'pdf_file_path' in locals() else 'Unknown'}")
    print(f"Total Pages: {results['total_pages']}")
    print(f"OCR Pages: {results['ocr_pages_count']}")
    if results['ocr_performed']:
        print(f"Average OCR Confidence: {results['avg_ocr_confidence']:.2f}")
    else:
        print("OCR Source: Direct text extraction was sufficient.")

    print("\n--- Extracted Text Sample (First 500 chars) ---")
    print(results['extracted_text'][:500] + "...")
else:
    print("No results found. Please ensure you have executed the processing cell (896afa14) after uploading your file.")

No results found. Please ensure you have executed the processing cell (896afa14) after uploading your file.


In [ ]:
from pypdf import PdfReader
from PIL import Image
import pytesseract
import io
from pdf2image import convert_from_path # Import for real OCR

def process_pdf_for_text_and_ocr_quality(pdf_path):
    """
    Extracts text from a PDF, counts pages, and attempts to assess OCR quality.

    Args:
        pdf_path (str): The path to the PDF file.

    Returns:
        dict: A dictionary containing:
            - 'total_pages': int, the total number of pages.
            - 'extracted_text': str, the full text extracted from the PDF.
            - 'ocr_pages_count': int, number of pages where OCR was likely used.
            - 'avg_ocr_confidence': float, average OCR confidence (0-100) if OCR was used, else 0.
            - 'details_per_page': list of dicts, details for each page including text and OCR info.
            - 'ocr_performed': bool, True if OCR was performed on at least one page.
    """
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)
    full_text = []
    ocr_confidences = []
    ocr_pages_count = 0
    details_per_page = []
    ocr_performed_overall = False

    for i, page in enumerate(reader.pages):
        page_text = page.extract_text()
        page_detail = {
            'page_number': i + 1,
            'text_source': 'direct_extraction',
            'text': page_text if page_text else '',
            'ocr_confidence': 0
        }

        # If direct text extraction is minimal or empty, try OCR
        if not page_text or len(page_text.strip()) < 50: # Arbitrary threshold for "minimal text"
            ocr_pages_count += 1
            ocr_performed_overall = True

            try:
                # Convert page to image for OCR using pdf2image
                images = convert_from_path(pdf_path, first_page=i + 1, last_page=i + 1)
                if images:
                    ocr_image = images[0]

                    # Perform OCR
                    ocr_text = pytesseract.image_to_string(ocr_image, lang='por') # Specify Portuguese language

                    # Get confidence scores
                    data = pytesseract.image_to_data(ocr_image, output_type=pytesseract.Output.DICT, lang='por')
                    confidences = [int(c) for c in data['conf'] if c.isdigit() and int(c) != -1]
                    avg_page_confidence = sum(confidences) / len(confidences) if confidences else 0
                else:
                    ocr_text = "OCR processing failed: No image converted for this page."
                    avg_page_confidence = 0
            except Exception as e:
                ocr_text = f"OCR processing error for Page {i+1}: {e}"
                avg_page_confidence = 0

            page_detail['text_source'] = 'ocr_used_real'
            page_detail['text'] = ocr_text.strip()
            page_detail['ocr_confidence'] = avg_page_confidence
            if avg_page_confidence > 0: # Only add valid confidences
                ocr_confidences.append(avg_page_confidence)

        full_text.append(page_detail['text'])
        details_per_page.append(page_detail)

    avg_ocr_confidence = sum(ocr_confidences) / len(ocr_confidences) if ocr_confidences else 0

    return {
        'total_pages': total_pages,
        'extracted_text': "\n".join(full_text),
        'ocr_pages_count': ocr_pages_count,
        'avg_ocr_confidence': avg_ocr_confidence,
        'details_per_page': details_per_page,
        'ocr_performed': ocr_performed_overall
    }

In [18]:
from pypdf import PdfReader
from PIL import Image
import pytesseract
import io
from pdf2image import convert_from_path # Import for real OCR

def process_pdf_for_text_and_ocr_quality(pdf_path):
    """
    Extracts text from a PDF, counts pages, and attempts to assess OCR quality.

    Args:
        pdf_path (str): The path to the PDF file.

    Returns:
        dict: A dictionary containing:
            - 'total_pages': int, the total number of pages.
            - 'extracted_text': str, the full text extracted from the PDF.
            - 'ocr_pages_count': int, number of pages where OCR was likely used.
            - 'avg_ocr_confidence': float, average OCR confidence (0-100) if OCR was used, else 0.
            - 'details_per_page': list of dicts, details for each page including text and OCR info.
            - 'ocr_performed': bool, True if OCR was performed on at least one page.
    """
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)
    full_text = []
    ocr_confidences = []
    ocr_pages_count = 0
    details_per_page = []
    ocr_performed_overall = False

    for i, page in enumerate(reader.pages):
        page_text = page.extract_text()
        page_detail = {
            'page_number': i + 1,
            'text_source': 'direct_extraction',
            'text': page_text if page_text else '',
            'ocr_confidence': 0
        }

        # If direct text extraction is minimal or empty, try OCR
        if not page_text or len(page_text.strip()) < 50: # Arbitrary threshold for "minimal text"
            ocr_pages_count += 1
            ocr_performed_overall = True

            try:
                # Convert page to image for OCR using pdf2image
                images = convert_from_path(pdf_path, first_page=i + 1, last_page=i + 1)
                if images:
                    ocr_image = images[0]

                    # Perform OCR
                    ocr_text = pytesseract.image_to_string(ocr_image, lang='por') # Specify Portuguese language

                    # Get confidence scores
                    data = pytesseract.image_to_data(ocr_image, output_type=pytesseract.Output.DICT, lang='por')
                    confidences = [int(c) for c in data['conf'] if c.isdigit() and int(c) != -1]
                    avg_page_confidence = sum(confidences) / len(confidences) if confidences else 0
                else:
                    ocr_text = "OCR processing failed: No image converted for this page."
                    avg_page_confidence = 0
            except Exception as e:
                ocr_text = f"OCR processing error for Page {i+1}: {e}"
                avg_page_confidence = 0

            page_detail['text_source'] = 'ocr_used_real'
            page_detail['text'] = ocr_text.strip()
            page_detail['ocr_confidence'] = avg_page_confidence
            if avg_page_confidence > 0: # Only add valid confidences
                ocr_confidences.append(avg_page_confidence)

        full_text.append(page_detail['text'])
        details_per_page.append(page_detail)

    avg_ocr_confidence = sum(ocr_confidences) / len(ocr_confidences) if ocr_confidences else 0

    return {
        'total_pages': total_pages,
        'extracted_text': "\n".join(full_text),
        'ocr_pages_count': ocr_pages_count,
        'avg_ocr_confidence': avg_ocr_confidence,
        'details_per_page': details_per_page,
        'ocr_performed': ocr_performed_overall
    }

In [19]:
from pypdf import PdfReader
from PIL import Image
import pytesseract
import io
from pdf2image import convert_from_path

def process_pdf_for_text_and_ocr_quality(pdf_path):
    """
    Extracts text from a PDF, counts pages, and performs OCR if necessary.
    Applies line numbering in the format Page.Line (e.g., 1.01).
    """
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)
    full_text_pages = []
    ocr_confidences = []
    ocr_pages_count = 0
    ocr_performed_overall = False

    for i in range(total_pages):
        page = reader.pages[i]
        page_text = page.extract_text()

        # Fallback to OCR if page is empty or mostly image
        if not page_text or len(page_text.strip()) < 50:
            ocr_pages_count += 1
            ocr_performed_overall = True
            try:
                images = convert_from_path(pdf_path, first_page=i+1, last_page=i+1, dpi=300)
                if images:
                    img = images[0]
                    page_text = pytesseract.image_to_string(img, lang='por')
                    data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT, lang='por')
                    conf_list = [int(c) for c in data['conf'] if str(c).isdigit() and int(c) != -1]
                    avg_conf = sum(conf_list) / len(conf_list) if conf_list else 0
                    ocr_confidences.append(avg_conf)
            except Exception as e:
                page_text = f"[Error processing page {i+1}: {e}]"

        # Process lines with numbering
        lines = page_text.splitlines()
        numbered_lines = []
        for line_idx, line in enumerate(lines, 1):
            if line.strip(): # Number non-empty lines
                line_label = f"{i+1}.{line_idx:02d}" & "|"
                numbered_lines.append(f"{line_label} {line}")
            else:
                numbered_lines.append("")

        full_text_pages.append("\n".join(numbered_lines))

    avg_ocr_confidence = sum(ocr_confidences) / len(ocr_confidences) if ocr_confidences else 0

    return {
        'total_pages': total_pages,
        'extracted_text': "\n\n--- PAGE BREAK ---\n\n".join(full_text_pages),
        'ocr_pages_count': ocr_pages_count,
        'avg_ocr_confidence': avg_ocr_confidence,
        'ocr_performed': ocr_performed_overall
    }

In [24]:
import os
from pypdf import PdfReader
from PIL import Image
import pytesseract
from pdf2image import convert_from_path

def process_pdf_final_v3(pdf_path):
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)
    full_text_pages = []
    ocr_confidences = []
    ocr_pages_count = 0

    print(f"Processing {total_pages} pages for: {os.path.basename(pdf_path)}")

    for i in range(total_pages):
        page = reader.pages[i]
        page_text = page.extract_text()

        # Fallback to OCR if page is empty or mostly image
        if not page_text or len(page_text.strip()) < 50:
            ocr_pages_count += 1
            try:
                images = convert_from_path(pdf_path, first_page=i+1, last_page=i+1, dpi=300)
                if images:
                    img = images[0]
                    page_text = pytesseract.image_to_string(img, lang='por')
                    data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT, lang='por')
                    conf_list = [int(c) for c in data['conf'] if str(c).isdigit() and int(c) != -1]
                    if conf_list: ocr_confidences.append(sum(conf_list) / len(conf_list))
            except Exception as e:
                page_text = f"[Error page {i+1}: {e}]"

        # Format: Page.Line | Text
        lines = [l for l in page_text.splitlines() if l.strip()]
        numbered_lines = []
        for line_idx, line in enumerate(lines, 1):
            line_label = f"{i+1}.{line_idx:02d} |"
            numbered_lines.append(f"{line_label} {line}")

        full_text_pages.append("\n".join(numbered_lines))

    final_text = "\n\n--- PAGE BREAK ---\n\n".join(full_text_pages)
    avg_conf = (sum(ocr_confidences)/len(ocr_confidences) if ocr_confidences else 0)
    return final_text, total_pages, ocr_pages_count, avg_conf

# Detect and process
pdf_files = [f for f in os.listdir('.') if f.lower().endswith('.pdf')]
if pdf_files:
    target = pdf_files[0]
    text, total, ocr_count, conf = process_pdf_final_v3(target)

    output_path = os.path.splitext(target)[0] + "_numbered_final.txt"
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(text)

    print(f"\nSuccess! Result saved to: {output_path}")
    print(f"Summary: {total} pages, {ocr_count} OCR pages, Avg Confidence: {conf:.2f}%")
    print("\n--- Preview ---\n" + "\n".join(text.splitlines()[:10]))
else:
    print("No PDF found. Please upload your file via the sidebar.")

No PDF found. Please upload your file via the sidebar.


In [25]:
import os
import pytesseract
from pypdf import PdfReader
from PIL import Image
from pdf2image import convert_from_path
from google.colab import files

def process_pdf_final(pdf_path):
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)
    full_text_pages = []
    ocr_confidences = []
    ocr_pages_count = 0

    print(f"Processing {total_pages} pages for file: {os.path.basename(pdf_path)}")

    for i in range(total_pages):
        page = reader.pages[i]
        page_text = page.extract_text()

        # Fallback to OCR if page is empty or mostly image
        if not page_text or len(page_text.strip()) < 50:
            ocr_pages_count += 1
            try:
                images = convert_from_path(pdf_path, first_page=i+1, last_page=i+1, dpi=300)
                if images:
                    img = images[0]
                    page_text = pytesseract.image_to_string(img, lang='por')
                    data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT, lang='por')
                    conf_list = [int(c) for c in data['conf'] if str(c).isdigit() and int(c) != -1]
                    if conf_list: ocr_confidences.append(sum(conf_list) / len(conf_list))
            except Exception as e:
                page_text = f"[Error page {i+1}: {e}]"

        # Process lines with numbering and separator '|' resetting per page
        lines = [l for l in page_text.splitlines() if l.strip()]
        numbered_lines = []
        for line_idx, line in enumerate(lines, 1):
            line_label = f"{i+1}.{line_idx:02d} |"
            numbered_lines.append(f"{line_label} {line}")

        full_text_pages.append("\n".join(numbered_lines))

    final_text = "\n\n--- PAGE BREAK ---\n\n".join(full_text_pages)
    return final_text, total_pages, ocr_pages_count, (sum(ocr_confidences)/len(ocr_confidences) if ocr_confidences else 0)

# Automatically find the uploaded PDF
pdf_files = [f for f in os.listdir('.') if f.lower().endswith('.pdf')]

if pdf_files:
    # Priority to the file mentioned in history, otherwise take the latest/first
    target_file = pdf_files[0]
    text, total, ocr_count, avg_conf = process_pdf_final(target_file)

    out_name = os.path.splitext(target_file)[0] + "_final_v2.txt"
    with open(out_name, "w", encoding="utf-8") as f:
        f.write(text)

    print(f"\nSuccess! Saved to: {out_name}")
    print(f"Summary: {total} pages, {ocr_count} used OCR, Avg Confidence: {avg_conf:.2f}%")
    print("\n--- Preview (First 10 lines) ---\n" + "\n".join(text.splitlines()[:10]))
else:
    print("No PDF file found in /content. Please upload a file using the sidebar.")

No PDF file found in /content. Please upload a file using the sidebar.


In [27]:
from google.colab import files
import os

# 1. Ask for file upload
print("Please upload the PDF file you wish to process:")
uploaded = files.upload()

if uploaded:
    # Get the name of the uploaded file
    pdf_filename = list(uploaded.keys())[0]
    print(f"\nProcessing file: {pdf_filename}...")

    # 2. Call the processing function
    # Note: process_pdf_final_v3 must be defined in the previous cell
    text_output, total_pgs, ocr_pgs, avg_conf = process_pdf_final_v3(pdf_filename)

    # 3. Save the result
    final_output_name = os.path.splitext(pdf_filename)[0] + "_numbered_result.txt"
    with open(final_output_name, "w", encoding="utf-8") as f:
        f.write(text_output)

    print(f"\n--- Success! ---")
    print(f"File saved as: {final_output_name}")
    print(f"Pages: {total_pgs} | OCR Used: {ocr_pgs} pages | Avg Confidence: {avg_conf:.2f}%")
    print("\n--- Preview (First 10 lines) ---")
    print("\n".join(text_output.splitlines()[:10]))
else:
    print("No file uploaded.")

Please upload the PDF file you wish to process:


Saving Claudio_Zana_Junior_27_03_2026_08_31 assinado.pdf to Claudio_Zana_Junior_27_03_2026_08_31 assinado.pdf

Processing file: Claudio_Zana_Junior_27_03_2026_08_31 assinado.pdf...
Processing 17 pages for: Claudio_Zana_Junior_27_03_2026_08_31 assinado.pdf

--- Success! ---
File saved as: Claudio_Zana_Junior_27_03_2026_08_31 assinado_numbered_result.txt
Pages: 17 | OCR Used: 17 pages | Avg Confidence: 93.78%

--- Preview (First 10 lines) ---
1.01 | Avaliação de Desempenho 2026
1.02 | Avaliação direta ou 90º
1.03 | Criada em Prazo de participação
1.04 | 05/01/2026 às 10:57 05/01/2026 - 31/03/2026
1.05 | | Claudio Zana Junior
1.06 | Consultor Técnico de Vendas Sr.
1.07 | Departamento Tempo de empresa
1.08 | VENDAS 12 anos e 3 meses
1.09 | Ativa
1.10 | Escala utilizada


In [28]:
# No Linux (Ubuntu/Debian):
# !sudo apt update
# !sudo apt install -y tesseract-ocr tesseract-ocr-por poppler-utils

# Bibliotecas Python necessárias:
!pip install pypdf Pillow pytesseract pdf2image

In [29]:
import os
from pypdf import PdfReader
from PIL import Image
import pytesseract
from pdf2image import convert_from_path

def process_pdf_final_v3(pdf_path):
    """
    L6e um PDF, aplica OCR se necess1rio e formata o texto como 'Pagina.Linha | Conte7do'.
    """
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)
    full_text_pages = []
    ocr_confidences = []
    ocr_pages_count = 0

    print(f"Processando {total_pages} p1ginas: {os.path.basename(pdf_path)}")

    for i in range(total_pages):
        page = reader.pages[i]
        page_text = page.extract_text()

        # Fallback para OCR se a p1gina estiver vazia ou for imagem
        if not page_text or len(page_text.strip()) < 50:
            ocr_pages_count += 1
            try:
                # Converte p1gina em imagem (requer Poppler instalado no sistema)
                images = convert_from_path(pdf_path, first_page=i+1, last_page=i+1, dpi=300)
                if images:
                    img = images[0]
                    # Executa OCR (requer Tesseract instalado no sistema)
                    page_text = pytesseract.image_to_string(img, lang='por')
                    data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT, lang='por')
                    conf_list = [int(c) for c in data['conf'] if str(c).isdigit() and int(c) != -1]
                    if conf_list: ocr_confidences.append(sum(conf_list) / len(conf_list))
            except Exception as e:
                page_text = f"[Erro na p1gina {i+1}: {e}]"

        # Formata17o: Pagina.Linha |
        lines = [l for l in page_text.splitlines() if l.strip()]
        numbered_lines = []
        for line_idx, line in enumerate(lines, 1):
            line_label = f"{i+1}.{line_idx:02d} |"
            numbered_lines.append(f"{line_label} {line}")

        full_text_pages.append("\n".join(numbered_lines))

    final_text = "\n\n--- PAGE BREAK ---\n\n".join(full_text_pages)
    avg_conf = (sum(ocr_confidences)/len(ocr_confidences) if ocr_confidences else 0)
    return final_text, total_pages, ocr_pages_count, avg_conf

In [30]:
from google.colab import files

# Upload do arquivo
print("Fa7a o upload do seu PDF:")
uploaded = files.upload()

if uploaded:
    pdf_filename = list(uploaded.keys())[0]

    # Processamento
    text_output, total_pgs, ocr_pgs, avg_conf = process_pdf_final_v3(pdf_filename)

    # Salva o resultado
    output_name = os.path.splitext(pdf_filename)[0] + "_final_formatado.txt"
    with open(output_name, "w", encoding="utf-8") as f:
        f.write(text_output)

    print(f"\nConclu1do: Arquivo '{output_name}' gerado com sucesso!")
    print(f"P1ginas: {total_pgs} | OCR: {ocr_pgs} | Confian7a: {avg_conf:.2f}%")

Fa7a o upload do seu PDF:


Saving 36474838850-IRPF-2026-2025-retif-imagem-recibo.pdf to 36474838850-IRPF-2026-2025-retif-imagem-recibo.pdf
Processando 2 p1ginas: 36474838850-IRPF-2026-2025-retif-imagem-recibo.pdf

Conclu1do: Arquivo '36474838850-IRPF-2026-2025-retif-imagem-recibo_final_formatado.txt' gerado com sucesso!
P1ginas: 2 | OCR: 0 | Confian7a: 0.00%


In [ ]:
# Install pdf2image Python library
!pip install pdf2image

# Install poppler-utils system package (required by pdf2image)
!sudo apt update
!sudo apt install poppler-utils

print("pdf2image and poppler-utils installed!")

### Integrando em uma Ferramenta Web Online

As funções e o código acima fornecem o 'core' do seu backend para processamento de PDF em Python. Para transformá-lo em uma ferramenta web online, você precisaria:

1.  **Framework Web**: Usar um framework como Flask ou Django (Python), Node.js (JavaScript), ou outro.
2.  **Interface de Usuário (Frontend)**: Criar páginas HTML/CSS/JavaScript para:
    *   Um botão de upload de arquivo (`<input type="file">`).
    *   Exibir os resultados (número de páginas, texto extraído, qualidade do OCR).
3.  **API (Backend)**: No seu servidor web (usando Flask, por exemplo):
    *   Criar uma rota (`/upload_pdf`) que recebe o arquivo PDF enviado pelo usuário.
    *   Salvar temporariamente o arquivo PDF no servidor.
    *   Chamar a função `process_pdf_for_text_and_ocr_quality` com o caminho do arquivo temporário.
    *   Retornar os resultados (o dicionário) como JSON para o frontend.
4.  **Implantação**: Implantar seu aplicativo web em um serviço de hospedagem que suporte aplicativos Python (como Google Cloud Run, Heroku, AWS Elastic Beanstalk, etc.).

Este ambiente Colab é excelente para desenvolver e testar a lógica central, mas a criação da interface web e a hospedagem são etapas adicionais de desenvolvimento web.